# 03 - Experiments

对比实验 + 完整 Benchmark。运行完成后所有结果自动保存到 `eval/results/` 目录。

**实验矩阵**:
1. **Ablation**: single_agent vs multi_agent_nomcp vs multi_agent_mcp
2. **Full Benchmark**: 8个任务完整评测 (no-mcp)
3. **Full Benchmark + MCP**: 8个任务完整评测 (with mcp)

**预估耗时**: ~30-40 分钟 (取决于 API 响应速度)
**预估 Token**: ~300K-500K (DeepSeek-chat)

In [ ]:
# Cell 1: 环境准备
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = "/content/drive/MyDrive/CodeAgent-MCP"
os.chdir(PROJECT_DIR)
print(f"Working dir: {os.getcwd()}")

!pip install -q openai mcp pydantic pyyaml rich nest_asyncio

In [ ]:
# Cell 2: API Key + 环境变量
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("DEEPSEEK_API_KEY")

with open("/tmp/.api_key", "w") as f:
    f.write(os.environ["OPENAI_API_KEY"])

print("API key configured")

---
## 实验 1: Ablation Study (B1 + B4 + B7)

对比三种配置在 3 个代表性任务上的表现：
- **single_agent**: 只有 Coder，无 Planner 拆解、无 Reviewer 审查
- **multi_agent_nomcp**: Planner + Coder + Reviewer，但 Coder 不用工具
- **multi_agent_mcp**: 完整系统，Coder 通过 MCP 写文件、跑测试

任务选择: B1(medium, LRU Cache), B4(easy, CSV分析), B7(easy, 配置管理器)

In [ ]:
# Cell 3: Ablation 实验脚本
%%writefile /tmp/run_ablation.py
import asyncio
import json
import os
import sys
import time
from datetime import datetime

os.chdir("/content/drive/MyDrive/CodeAgent-MCP")
sys.path.insert(0, ".")

with open("/tmp/.api_key") as f:
    os.environ["OPENAI_API_KEY"] = f.read().strip()

# MCP workspace
WORKSPACE = "/tmp/workspace_ablation"
os.makedirs(WORKSPACE, exist_ok=True)
os.environ["FILE_SERVER_ROOT"] = WORKSPACE
os.environ["SHELL_SERVER_CWD"] = WORKSPACE
os.environ["GIT_SERVER_ROOT"] = WORKSPACE

from src.core.config import load_settings, load_agents_config, load_mcp_config
from src.core.llm_client import LLMClient
from src.core.orchestrator import Orchestrator
from src.agents import PlannerAgent, CoderAgent, ReviewerAgent
from src.mcp.client import MCPManager
from eval.run_eval import load_benchmark, run_single_task


async def run_single_agent(task, settings, agents_config, provider):
    """Single agent: Coder only, no Planner/Reviewer."""
    llm = LLMClient.from_settings(provider, settings)
    coder = CoderAgent(agents_config["coder"], llm, mcp_manager=None)
    start = time.time()
    try:
        code = await coder.run(f"请完成以下编程任务，输出完整可运行的 Python 代码：\n{task['description']}")
        elapsed = time.time() - start
        has_class = 'class ' in code
        has_tests = 'def test_' in code or 'assert ' in code
        return {
            "task_id": task["task_id"],
            "task_name": task["name"],
            "difficulty": task["difficulty"],
            "status": "completed",
            "review_score": None,
            "attempts": 1,
            "total_tokens": coder.total_tokens_used,
            "elapsed_seconds": round(elapsed, 1),
            "code_checks": {"has_class": has_class, "has_tests": has_tests},
            "code_length": len(code),
        }
    except Exception as e:
        return {
            "task_id": task["task_id"],
            "task_name": task["name"],
            "status": "error",
            "error": str(e),
            "elapsed_seconds": round(time.time() - start, 1),
        }


async def main():
    tasks = load_benchmark()
    selected_ids = ["B1", "B4", "B7"]
    tasks = [t for t in tasks if t["task_id"] in selected_ids]
    print(f"Tasks: {[t['task_id'] + ' ' + t['name'] for t in tasks]}")

    settings = load_settings()
    agents_config = load_agents_config()
    mcp_config = load_mcp_config()

    all_results = {}

    # --- Config 1: Single Agent ---
    print("\n" + "="*60)
    print("Config: single_agent_nomcp")
    print("="*60)
    results = []
    for t in tasks:
        print(f"  Running {t['task_id']}: {t['name']}...")
        r = await run_single_agent(t, settings, agents_config, "default")
        results.append(r)
        print(f"    -> {r['status']}, tokens={r.get('total_tokens', '?')}")
    all_results["single_agent_nomcp"] = results

    # --- Config 2: Multi Agent, no MCP ---
    print("\n" + "="*60)
    print("Config: multi_agent_nomcp")
    print("="*60)
    results = []
    for t in tasks:
        print(f"  Running {t['task_id']}: {t['name']}...")
        r = await run_single_task(t, settings, agents_config, mcp_config, "default", use_mcp=False)
        results.append(r)
        print(f"    -> {r['status']}, score={r.get('review_score', 'N/A')}, tokens={r.get('total_tokens', '?')}")
    all_results["multi_agent_nomcp"] = results

    # --- Config 3: Multi Agent + MCP ---
    print("\n" + "="*60)
    print("Config: multi_agent_mcp")
    print("="*60)
    results = []
    for t in tasks:
        # 每个任务清理 workspace
        import shutil
        for item in os.listdir(WORKSPACE):
            p = os.path.join(WORKSPACE, item)
            if os.path.isdir(p):
                shutil.rmtree(p, ignore_errors=True)
            else:
                os.remove(p)
        print(f"  Running {t['task_id']}: {t['name']}...")
        r = await run_single_task(t, settings, agents_config, mcp_config, "default", use_mcp=True)
        # 记录 workspace 产物
        ws_files = [f for f in os.listdir(WORKSPACE) if os.path.isfile(os.path.join(WORKSPACE, f))]
        r["workspace_files"] = ws_files
        results.append(r)
        print(f"    -> {r['status']}, score={r.get('review_score', 'N/A')}, tokens={r.get('total_tokens', '?')}, files={ws_files}")
    all_results["multi_agent_mcp"] = results

    # --- 输出汇总 ---
    print("\n" + "="*80)
    print("ABLATION RESULTS")
    print("="*80)
    print(f"\n{'Config':<25} {'Completion':<12} {'Avg Score':<12} {'Avg Tokens':<12} {'Avg Time':<10}")
    print("-" * 75)
    for name, results in all_results.items():
        total = len(results)
        completed = sum(1 for r in results if r['status'] == 'completed')
        scores = [r['review_score'] for r in results if r.get('review_score') is not None]
        tokens = [r['total_tokens'] for r in results if r.get('total_tokens')]
        times = [r['elapsed_seconds'] for r in results if r.get('elapsed_seconds')]
        avg_s = f"{sum(scores)/len(scores):.1f}" if scores else "N/A"
        avg_t = f"{sum(tokens)/len(tokens):.0f}" if tokens else "N/A"
        avg_e = f"{sum(times)/len(times):.1f}s" if times else "N/A"
        print(f"{name:<25} {completed}/{total:<10} {avg_s:<12} {avg_t:<12} {avg_e:<10}")

    # --- 按任务对比 ---
    print(f"\n{'Task':<12} {'single_agent':<20} {'multi_nomcp':<20} {'multi_mcp':<20}")
    print("-" * 75)
    for i, t in enumerate(tasks):
        cols = []
        for name in ["single_agent_nomcp", "multi_agent_nomcp", "multi_agent_mcp"]:
            r = all_results[name][i]
            score = r.get('review_score', '-')
            tokens = r.get('total_tokens', 0)
            cols.append(f"s={score} t={tokens}")
        print(f"{t['task_id']:<12} {cols[0]:<20} {cols[1]:<20} {cols[2]:<20}")

    # --- 保存 JSON ---
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output = {
        "experiment": "ablation",
        "timestamp": timestamp,
        "tasks": selected_ids,
        "configs": {},
    }
    for name, results in all_results.items():
        scores = [r.get('review_score', 0) for r in results if r.get('review_score') is not None]
        tokens = [r.get('total_tokens', 0) for r in results]
        output["configs"][name] = {
            "completed": sum(1 for r in results if r['status'] == 'completed'),
            "total": len(results),
            "avg_score": round(sum(scores)/len(scores), 1) if scores else None,
            "avg_tokens": round(sum(tokens)/len(tokens)) if tokens else None,
            "results": results,
        }

    os.makedirs("eval/results", exist_ok=True)
    path = f"eval/results/ablation_{timestamp}.json"
    with open(path, "w", encoding="utf-8") as f:
        json.dump(output, f, indent=2, ensure_ascii=False)
    print(f"\nResults saved to: {path}")

asyncio.run(main())

In [ ]:
!python /tmp/run_ablation.py

---
## 实验 2: Full Benchmark (8 tasks, no-mcp)

跑全部 8 个 benchmark 任务，测试多 Agent 编排器在不同难度任务上的表现。

In [ ]:
# Cell 5: Full benchmark 脚本
%%writefile /tmp/run_full_benchmark.py
import asyncio
import json
import os
import sys
import time
from datetime import datetime

os.chdir("/content/drive/MyDrive/CodeAgent-MCP")
sys.path.insert(0, ".")

with open("/tmp/.api_key") as f:
    os.environ["OPENAI_API_KEY"] = f.read().strip()

from eval.run_eval import load_benchmark, run_single_task, print_summary, save_results
from src.core.config import load_settings, load_agents_config, load_mcp_config

async def main():
    tasks = load_benchmark()
    settings = load_settings()
    agents_config = load_agents_config()
    mcp_config = load_mcp_config()

    print(f"Running full benchmark: {len(tasks)} tasks (no-mcp)")
    print("-" * 50)

    results = []
    for i, task in enumerate(tasks):
        print(f"\n[{i+1}/{len(tasks)}] {task['task_id']}: {task['name']} ({task['difficulty']})")
        result = await run_single_task(
            task, settings, agents_config, mcp_config,
            provider="default", use_mcp=False,
        )
        results.append(result)
        score = result.get('review_score', 'N/A')
        tokens = result.get('total_tokens', '?')
        print(f"  -> {result['status']} | score={score} | tokens={tokens} | time={result.get('elapsed_seconds', '?')}s")

    print_summary(results)
    save_results(results, "eval/results")

    # 按难度分组统计
    print("\nBy difficulty:")
    for diff in ["easy", "medium", "hard"]:
        group = [r for r in results if r.get('difficulty') == diff]
        if group:
            completed = sum(1 for r in group if r['status'] == 'completed')
            scores = [r['review_score'] for r in group if r.get('review_score') is not None]
            avg = sum(scores)/len(scores) if scores else 0
            print(f"  {diff}: {completed}/{len(group)} completed, avg score={avg:.1f}")

asyncio.run(main())

In [ ]:
!python /tmp/run_full_benchmark.py

---
## 实验 3: Full Benchmark + MCP (8 tasks)

同样 8 个任务，但启用 MCP 工具，Coder 可以写文件、跑测试。

**注意**: MCP 模式 token 消耗更高（Coder 需要多轮工具调用），预估 ~400K tokens。

In [ ]:
# Cell 7: Full benchmark + MCP 脚本
%%writefile /tmp/run_full_benchmark_mcp.py
import asyncio
import json
import os
import shutil
import sys
import time
from datetime import datetime

os.chdir("/content/drive/MyDrive/CodeAgent-MCP")
sys.path.insert(0, ".")

with open("/tmp/.api_key") as f:
    os.environ["OPENAI_API_KEY"] = f.read().strip()

WORKSPACE = "/tmp/workspace_bench"
os.makedirs(WORKSPACE, exist_ok=True)
os.environ["FILE_SERVER_ROOT"] = WORKSPACE
os.environ["SHELL_SERVER_CWD"] = WORKSPACE
os.environ["GIT_SERVER_ROOT"] = WORKSPACE

from eval.run_eval import load_benchmark, run_single_task, print_summary, save_results
from src.core.config import load_settings, load_agents_config, load_mcp_config

async def main():
    tasks = load_benchmark()
    settings = load_settings()
    agents_config = load_agents_config()
    mcp_config = load_mcp_config()

    print(f"Running full benchmark: {len(tasks)} tasks (WITH MCP)")
    print("-" * 50)

    results = []
    for i, task in enumerate(tasks):
        # 每个任务清理 workspace
        for item in os.listdir(WORKSPACE):
            p = os.path.join(WORKSPACE, item)
            if os.path.isdir(p):
                shutil.rmtree(p, ignore_errors=True)
            else:
                os.remove(p)

        print(f"\n[{i+1}/{len(tasks)}] {task['task_id']}: {task['name']} ({task['difficulty']})")
        result = await run_single_task(
            task, settings, agents_config, mcp_config,
            provider="default", use_mcp=True,
        )

        # 记录 workspace 产物
        ws_files = [f for f in os.listdir(WORKSPACE) if os.path.isfile(os.path.join(WORKSPACE, f))]
        result["workspace_files"] = ws_files
        results.append(result)

        score = result.get('review_score', 'N/A')
        tokens = result.get('total_tokens', '?')
        print(f"  -> {result['status']} | score={score} | tokens={tokens} | files={ws_files}")

    print_summary(results)

    # 保存结果
    os.makedirs("eval/results", exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    path = f"eval/results/eval_default_mcp_{timestamp}.json"
    with open(path, "w", encoding="utf-8") as f:
        json.dump({
            "timestamp": timestamp,
            "config": {"provider": "default", "use_mcp": True},
            "summary": {
                "total": len(results),
                "completed": sum(1 for r in results if r['status'] == 'completed'),
                "avg_score": sum(r.get('review_score', 0) for r in results) / len(results),
                "total_tokens": sum(r.get('total_tokens', 0) for r in results),
            },
            "results": results,
        }, f, indent=2, ensure_ascii=False)
    print(f"\nResults saved to: {path}")

    # 按难度分组
    print("\nBy difficulty:")
    for diff in ["easy", "medium", "hard"]:
        group = [r for r in results if r.get('difficulty') == diff]
        if group:
            completed = sum(1 for r in group if r['status'] == 'completed')
            scores = [r['review_score'] for r in group if r.get('review_score') is not None]
            avg = sum(scores)/len(scores) if scores else 0
            print(f"  {diff}: {completed}/{len(group)} completed, avg score={avg:.1f}")

asyncio.run(main())

In [ ]:
!python /tmp/run_full_benchmark_mcp.py

---
## 汇总对比

In [ ]:
# Cell 9: 汇总所有实验结果
import json
import os
from pathlib import Path

results_dir = "eval/results"
print("Available result files:")
for f in sorted(os.listdir(results_dir)):
    if f.endswith(".json"):
        path = os.path.join(results_dir, f)
        size = os.path.getsize(path)
        print(f"  {f} ({size} bytes)")

# 加载最新的各类结果
def load_latest(prefix):
    files = sorted([f for f in os.listdir(results_dir) if f.startswith(prefix) and f.endswith(".json")])
    if files:
        with open(os.path.join(results_dir, files[-1]), encoding="utf-8") as f:
            return json.load(f), files[-1]
    return None, None

ablation, abl_file = load_latest("ablation_")
nomcp, nomcp_file = load_latest("eval_default_nomcp_")
mcp, mcp_file = load_latest("eval_default_mcp_")

print("\n" + "="*80)
print("EXPERIMENT SUMMARY")
print("="*80)

if ablation:
    print(f"\n--- Ablation ({abl_file}) ---")
    for name, cfg in ablation.get("configs", {}).items():
        print(f"  {name}: {cfg['completed']}/{cfg['total']} completed, "
              f"avg_score={cfg.get('avg_score', 'N/A')}, avg_tokens={cfg.get('avg_tokens', 'N/A')}")

if nomcp:
    s = nomcp["summary"]
    print(f"\n--- Full Benchmark no-mcp ({nomcp_file}) ---")
    print(f"  {s['completed']}/{s['total']} completed, avg_score={s['avg_score']:.1f}, total_tokens={s['total_tokens']}")

if mcp:
    s = mcp["summary"]
    print(f"\n--- Full Benchmark MCP ({mcp_file}) ---")
    print(f"  {s['completed']}/{s['total']} completed, avg_score={s['avg_score']:.1f}, total_tokens={s['total_tokens']}")

# 生成 Markdown 表格供 README 使用
print("\n" + "="*80)
print("MARKDOWN TABLE (copy to README.md)")
print("="*80)
print("\n| Configuration | Tasks | Completion | Avg Score | Total Tokens |")
print("|---|---|---|---|---|")
if ablation:
    for name, cfg in ablation["configs"].items():
        print(f"| {name} | {ablation['tasks']} | {cfg['completed']}/{cfg['total']} | {cfg.get('avg_score', 'N/A')} | {cfg.get('avg_tokens', 'N/A')} |")
if nomcp:
    s = nomcp["summary"]
    print(f"| multi_agent_nomcp (full) | B1-B8 | {s['completed']}/{s['total']} | {s['avg_score']:.1f} | {s['total_tokens']} |")
if mcp:
    s = mcp["summary"]
    print(f"| multi_agent_mcp (full) | B1-B8 | {s['completed']}/{s['total']} | {s['avg_score']:.1f} | {s['total_tokens']} |")

---
## 运行指南

1. **按顺序运行 Cell 1-2** (环境准备)
2. **运行 Cell 3-4** (Ablation, ~10-15分钟)
3. **运行 Cell 5-6** (Full Benchmark no-mcp, ~15分钟)
4. **(可选) 运行 Cell 7-8** (Full Benchmark MCP, ~20分钟, token 消耗更高)
5. **运行 Cell 9** (汇总对比, 生成 README 表格)

如果 token 预算有限，可以只跑 Cell 3-4 (Ablation) + Cell 9 (汇总)。

所有结果自动保存到 `eval/results/` 目录，同步到本地后可供 Claude Code 读取分析。